# Emotion Recognition - Image Modality
## Pipeline: Data Cleaning → Train/Val/Test Split → DINOv2 Fine-tuning


Dataset: 9 Facial Expressions for YOLO


YOLO: You Only Look Once

Each label file txt contains:

| class\_id |   x\_center   |   y\_center  |     width    |    height    |
| :-------: | :-----------: | :----------: | :----------: | :----------: |
|    0-8    |    0.0-1.0    |    0.0-1.0   |    0.0-1.0   |    0.0-1.0   |
| (emotion) | (% from left) | (% from top) | (% of image) | (% of image) |

|        Measurement       | Meaning                                              |
| :----------------------: | :--------------------------------------------------- |
| **x\_center, y\_center** | Where the **face center** is located (as % of image) |
|     **width, height**    | How big the **face** is (as % of image)              |


| Class ID | Emotion   | Emoji |
| :------: | :-------- | :---: |
|     0    | anger     |   😠  |
|     1    | contempt  |   😒  |
|     2    | disgust   |   🤢  |
|     3    | fear      |   😨  |
|     4    | happy     |   😊  |
|     5    | neutral   |   😐  |
|     6    | sad       |   😢  |
|     7    | sleepy    |   😴  |
|     8    | surprised |   😲  |


While the dataset is diverse, class imbalance may exist.
Some expressions (e.g., "Contempt" or "Sleepy") may have fewer samples than common expressions like "Happy" or "Sad".

How the images match the labels? -> They have the same file name except the images are in jpg and labels are in txt.


## Class Reduction: 9 → 7 Classes (Dataset Rebuild Strategy)

**Approach:** Filter train/val/test splits to remove contempt & sleepy, then remap class IDs.

### Kept Classes (Original IDs Unchanged)

| Class ID | Emotion   | Emoji | Status |
|:--------:|:----------|:-----:|:------:|
| 0        | anger     | 😠    | ✅ keep |
| 2        | disgust   | 🤢    | ✅ keep |
| 3        | fear      | 😨    | ✅ keep |
| 4        | happy     | 😊    | ✅ keep |
| 5        | neutral   | 😐    | ✅ keep |
| 6        | sad       | 😢    | ✅ keep |
| 8        | surprised | 😲    | ✅ keep |

### Removed Classes

| Class ID | Emotion  | Emoji | Status  |
|:--------:|:---------|:-----:|:-------:|
| 1        | contempt | 😒    | ❌ purge |
| 7        | sleepy   | 😴    | ❌ purge |

---




**Emotions kept (7):** anger, disgust, fear, happy, neutral, sad, surprise  
**Emotions removed:** contempt, sleepy

## Step 0 — Check class IDs in your dataset
Run this first to confirm which IDs map to contempt and sleepy.

In [30]:
import os
import yaml

# ---  PATHS ---
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATASET_ROOT = os.path.join(project_root, "src", "data")
TRAIN_IMAGES = os.path.join(DATASET_ROOT, "train", "images")
TRAIN_LABELS = os.path.join(DATASET_ROOT, "train", "labels")
VAL_IMAGES   = os.path.join(DATASET_ROOT, "valid", "images")
VAL_LABELS   = os.path.join(DATASET_ROOT, "valid", "labels")
TEST_IMAGES  = os.path.join(DATASET_ROOT, "test", "images")
TEST_LABELS  = os.path.join(DATASET_ROOT, "test", "labels")

# Try loading data.yaml to find class names
yaml_path = os.path.join(DATASET_ROOT, "data.yaml")
if os.path.exists(yaml_path):
    with open(yaml_path) as f:
        data_cfg = yaml.safe_load(f)
    class_names = data_cfg.get("names", [])
    print("Classes from data.yaml:")
    for i, name in enumerate(class_names):
        print(f"  ID {i}: {name}")
else:
    print("data.yaml not found — check for classes.txt or list manually")
    # Fallback: inspect a few label files
    sample_labels = os.listdir(TRAIN_LABELS)[:5]
    for lf in sample_labels:
        print(f"\n{lf}:")
        with open(os.path.join(TRAIN_LABELS, lf)) as f:
            print(f.read())

Classes from data.yaml:
  ID 0: angry
  ID 1: contempt
  ID 2: disgust
  ID 3: fear
  ID 4: happy
  ID 5: natural
  ID 6: sad
  ID 7: sleepy
  ID 8: surprised


In [ ]:
# Will remove class id 1 and 7 which are contempt and sleepy

REMOVE_CLASS_IDS = {1, 7}   # <-- update with actual contempt and sleepy IDs
print(f"Will remove samples with class IDs: {REMOVE_CLASS_IDS}")

Will remove samples with class IDs: {1, 7}


## Step 1 — Clean Train, Val, and Test sets (remove contempt & sleepy)

In [ ]:
import os
import shutil
from pathlib import Path

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_image_for_label(label_path: Path, images_dir: Path) -> Path | None:
    """Find the corresponding image file for a given label file."""
    stem = label_path.stem
    for ext in IMAGE_EXTENSIONS:
        candidate = images_dir / (stem + ext)
        if candidate.exists():
            return candidate
    return None

def label_contains_removed_class(label_path: Path, remove_ids: set) -> bool:
    """Return True if any line in the YOLO label file has a class ID to remove."""
    with open(label_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            class_id = int(line.split()[0])
            if class_id in remove_ids:
                return True
    return False

def clean_split(images_dir: str, labels_dir: str, remove_ids: set, dry_run: bool = True):
    """
    Remove label files (and corresponding images) that contain any of the remove_ids.
    Set dry_run=True first to preview what will be deleted.
    """
    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)

    label_files = list(labels_dir.glob("*.txt"))
    print(f"Total label files: {len(label_files)}")

    removed_labels = []
    removed_images = []
    missing_images = []

    for lf in label_files:
        if label_contains_removed_class(lf, remove_ids):
            removed_labels.append(lf)
            img = find_image_for_label(lf, images_dir)
            if img:
                removed_images.append(img)
            else:
                missing_images.append(lf.stem)

    print(f"Labels to remove: {len(removed_labels)}")
    print(f"Images to remove: {len(removed_images)}")
    if missing_images:
        print(f"WARNING — no image found for {len(missing_images)} labels: {missing_images[:5]}")

    if dry_run:
        print("\nDRY RUN — nothing deleted. Set dry_run=False to actually delete.")
        print("Sample labels that would be removed:", [lf.name for lf in removed_labels[:5]])
    else:
        for lf in removed_labels:
            lf.unlink()
        for img in removed_images:
            img.unlink()
        print(f"Deleted {len(removed_labels)} labels and {len(removed_images)} images.")

    return len(label_files) - len(removed_labels)  # remaining count

# --- DRY RUN FIRST ---
# Dry run scans our preview files and shows us how many files would be deleted before ACTUALLY deleting the files~
print("=== TRAIN ===")
clean_split(TRAIN_IMAGES, TRAIN_LABELS, REMOVE_CLASS_IDS, dry_run=True)

print("\n=== VALIDATION ===")
clean_split(VAL_IMAGES, VAL_LABELS, REMOVE_CLASS_IDS, dry_run=True)

print("\n=== TEST ===")
clean_split(TEST_IMAGES, TEST_LABELS, REMOVE_CLASS_IDS, dry_run=True)

=== TRAIN ===
Total label files: 64866
Labels to remove: 3594
Images to remove: 3594

DRY RUN — nothing deleted. Set dry_run=False to actually delete.
Sample labels that would be removed: ['105_jpg.rf.c7a7b226ca235fd199cbf0d3f89faf60.txt', '1111_jpg.rf.4e51915a2cd7c4bb7644d0e4349b9f71.txt', '1112_jpg.rf.790f5a2c440967e1a12cdc608345d72b.txt', '1113_jpg.rf.8d9eb8b60df98dbef3fc0f19956c6fd0.txt', '1114_jpg.rf.97a7f9a5d46dfaeb58739b0b3e6282fd.txt']

=== VALIDATION ===
Total label files: 1720
Labels to remove: 120
Images to remove: 120

DRY RUN — nothing deleted. Set dry_run=False to actually delete.
Sample labels that would be removed: ['1172_jpg.rf.ef32da8c66206c8d8d37b78c52143076.txt', '219_jpg.rf.e93512e56d53ffc542c58b3c4270ef88.txt', '252_jpg.rf.a0042be227e42765015b5a1a91e64cab.txt', '253_jpg.rf.b14d89bcabf9fc091a60837a08ddfd90.txt', '265_jpg.rf.f6b4489d12cb4a4674f28eb9859d9695.txt']

=== TEST ===
Total label files: 1700
Labels to remove: 101
Images to remove: 101

DRY RUN — nothing del

1599

In [33]:
# --- RUN FOR REAL (only after dry run looks correct) ---
print("=== TRAIN ===")
remaining_train = clean_split(TRAIN_IMAGES, TRAIN_LABELS, REMOVE_CLASS_IDS, dry_run=False)

print("\n=== VALIDATION ===")
remaining_val = clean_split(VAL_IMAGES, VAL_LABELS, REMOVE_CLASS_IDS, dry_run=False)

print("\n=== TEST ===")
remaining_test = clean_split(TEST_IMAGES, TEST_LABELS, REMOVE_CLASS_IDS, dry_run=False)

print(f"\nRemaining: {remaining_train} train, {remaining_val} val, {remaining_test} test samples")

=== TRAIN ===
Total label files: 64866
Labels to remove: 3594
Images to remove: 3594
Deleted 3594 labels and 3594 images.

=== VALIDATION ===
Total label files: 1720
Labels to remove: 120
Images to remove: 120
Deleted 120 labels and 120 images.

=== TEST ===
Total label files: 1700
Labels to remove: 101
Images to remove: 101
Deleted 101 labels and 101 images.

Remaining: 61272 train, 1600 val, 1599 test samples


## Step 2 — Remap class IDs after removal
After removing contempt and sleepy, the remaining 7 class IDs need to be remapped to 0–6 consecutively, otherwise your model will see gaps (e.g. no class 1 or 7).

In [34]:
# Build a remapping from old ID -> new ID
# Example: original 9 classes, remove IDs 1 (contempt) and 7 (sleepy)
# Remaining old IDs: [0, 2, 3, 4, 5, 6, 8] -> new IDs: [0, 1, 2, 3, 4, 5, 6]

# Get ALL original class IDs from data.yaml
all_ids = set(range(len(class_names)))  # e.g. {0,1,2,3,4,5,6,7,8}
kept_ids = sorted(all_ids - REMOVE_CLASS_IDS)  # e.g. [0,2,3,4,5,6,8]
id_remap = {old: new for new, old in enumerate(kept_ids)}

print("Class ID remapping (old -> new):")
for old, new in id_remap.items():
    print(f"  {old} ({class_names[old]}) -> {new}")

NEW_CLASS_NAMES = [class_names[i] for i in kept_ids]
print(f"\nNew class list: {NEW_CLASS_NAMES}")

Class ID remapping (old -> new):
  0 (angry) -> 0
  2 (disgust) -> 1
  3 (fear) -> 2
  4 (happy) -> 3
  5 (natural) -> 4
  6 (sad) -> 5
  8 (surprised) -> 6

New class list: ['angry', 'disgust', 'fear', 'happy', 'natural', 'sad', 'surprised']


In [35]:
def remap_labels_in_dir(labels_dir: str, id_remap: dict):
    """Rewrite all label files in-place with remapped class IDs."""
    labels_dir = Path(labels_dir)
    label_files = list(labels_dir.glob("*.txt"))
    print(f"Remapping {len(label_files)} files in {labels_dir}")
    
    for lf in label_files:
        lines = lf.read_text().strip().splitlines()
        new_lines = []
        for line in lines:
            if not line.strip():
                continue
            parts = line.split()
            old_id = int(parts[0])
            if old_id not in id_remap:
                # Shouldn't happen after cleaning, but skip just in case
                continue
            parts[0] = str(id_remap[old_id])
            new_lines.append(" ".join(parts))
        lf.write_text("\n".join(new_lines) + "\n")
    
    print("Done.")

remap_labels_in_dir(TRAIN_LABELS, id_remap)
remap_labels_in_dir(VAL_LABELS, id_remap)
remap_labels_in_dir(TEST_LABELS, id_remap)

Remapping 61272 files in c:\Users\Admin\Desktop\CDS\src\data\train\labels
Done.
Remapping 1600 files in c:\Users\Admin\Desktop\CDS\src\data\valid\labels
Done.
Remapping 1599 files in c:\Users\Admin\Desktop\CDS\src\data\test\labels
Done.


## Step 3 — Verify dataset balance

In [ ]:
from collections import Counter
from pathlib import Path

# Get the distribution of emotions in the training set
c = Counter()
for lf in Path(TRAIN_LABELS).glob("*.txt"):
    line = lf.read_text().strip().splitlines()
    if line:
        c[int(line[0].split()[0])] += 1

print(c)

Counter({3: 13831, 5: 11969, 0: 11195, 6: 8960, 4: 5663, 2: 5358, 1: 4296})


In [37]:
from collections import Counter

def count_classes(labels_dir: str, class_names: list) -> Counter:
    counts = Counter()
    for lf in Path(labels_dir).glob("*.txt"):
        for line in lf.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1
    return counts

for split_name, ldir in [("Train", TRAIN_LABELS), ("Val", VAL_LABELS), ("Test", TEST_LABELS)]:
    counts = count_classes(ldir, NEW_CLASS_NAMES)
    total = sum(counts.values())
    print(f"\n{split_name} ({total} samples):")
    for cid in sorted(counts):
        name = NEW_CLASS_NAMES[cid] if cid < len(NEW_CLASS_NAMES) else f"id_{cid}"
        print(f"  {name}: {counts[cid]}")


Train (61272 samples):
  angry: 11195
  disgust: 4296
  fear: 5358
  happy: 13831
  natural: 5663
  sad: 11969
  surprised: 8960

Val (1600 samples):
  angry: 258
  disgust: 108
  fear: 107
  happy: 387
  natural: 172
  sad: 312
  surprised: 256

Test (1599 samples):
  angry: 297
  disgust: 98
  fear: 144
  happy: 407
  natural: 136
  sad: 298
  surprised: 219


## Step 4 — PyTorch Dataset for YOLO-format data

In [38]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path

class YOLOEmotionDataset(Dataset):
    """
    Loads images whose labels are in YOLO format.
    Each image gets the class of its first annotation.
    (For face emotion datasets, typically 1 face per image.)
    """
    def __init__(self, images_dir: str, labels_dir: str, transform=None):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.transform = transform
        self.samples = []  # list of (image_path, class_id)

        for lf in sorted(self.labels_dir.glob("*.txt")):
            lines = [l.strip() for l in lf.read_text().splitlines() if l.strip()]
            if not lines:
                continue
            class_id = int(lines[0].split()[0])  # use first annotation
            img = find_image_for_label(lf, self.images_dir)
            if img:
                self.samples.append((img, class_id))

        print(f"Loaded {len(self.samples)} samples from {images_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

DINOV2 Model:

Phase 1 (frozen backbone): Only the small head trains. This is fast and prevents your randomly-initialised head from immediately corrupting DINOv2's carefully pretrained weights with large random gradients. You let the head first learn a reasonable mapping from DINOv2's features to your 7 emotions.
Phase 2 (unfrozen backbone): Now that the head is stable, you unfreeze the whole backbone and fine-tune everything together with a much lower learning rate (LR/10). This lets DINOv2 subtly adjust its features specifically for facial emotion recognition.

## Step 5 — DINOv2 Fine-tuning

In [39]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from transformers import AutoModel
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

NUM_CLASSES = len(NEW_CLASS_NAMES)  # 7
BATCH_SIZE = 32
LR = 1e-4
EPOCHS = 10

# DINOv2 expects 224x224, normalized with ImageNet stats
dinov2_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_aug = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_ds = YOLOEmotionDataset(TRAIN_IMAGES, TRAIN_LABELS, transform=train_aug)
val_ds   = YOLOEmotionDataset(VAL_IMAGES,   VAL_LABELS,   transform=dinov2_transform)
test_ds  = YOLOEmotionDataset(TEST_IMAGES,  TEST_LABELS,  transform=dinov2_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True) #changed num_workers=0 due to jupyternotebook
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

Device: cuda
Loaded 61270 samples from c:\Users\Admin\Desktop\CDS\src\data\train\images
Loaded 1600 samples from c:\Users\Admin\Desktop\CDS\src\data\valid\images
Loaded 1599 samples from c:\Users\Admin\Desktop\CDS\src\data\test\images


In [40]:
class DINOv2Classifier(nn.Module):
    """
    DINOv2 backbone + linear classification head.
    We freeze the backbone initially and only train the head,
    then unfreeze for full fine-tuning.
    """
    def __init__(self, num_classes: int, backbone: str = "facebook/dinov2-base", freeze_backbone: bool = True):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(backbone)
        hidden_size = self.backbone.config.hidden_size  # 768 for base, 1024 for large
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, num_classes)
        )
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True
        print("Backbone unfrozen for full fine-tuning.")

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values)
        # Use CLS token (first token of last hidden state)
        cls_token = outputs.last_hidden_state[:, 0]
        return self.classifier(cls_token)


model = DINOv2Classifier(num_classes=NUM_CLASSES, freeze_backbone=True)
model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 5680.44it/s]


Trainable params: 6,919 / 86,587,399


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def run_epoch(model, loader, optimizer=None, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    
    with torch.set_grad_enabled(train):
        for images, labels in tqdm(loader, desc="Train" if train else "Eval", leave=False):
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            
            total_loss += loss.item()
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_preds, all_labels


best_val_acc = 0
# saves my learned weights from the best model
CHECKPOINT = "best_dinov2_emotion.pt"

# --- Phase 1: Train head only (frozen backbone) ---
print("Phase 1: Training classification head (backbone frozen)")
for epoch in range(min(3, EPOCHS)):
    train_loss, train_acc, _, _ = run_epoch(model, train_loader, optimizer, train=True)
    val_loss, val_acc, _, _ = run_epoch(model, val_loader, train=False)
    scheduler.step()
    
    print(f"Epoch {epoch+1:2d} | Train loss: {train_loss:.4f} acc: {train_acc:.4f} | Val loss: {val_loss:.4f} acc: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT)
        print(f"           ✓ Saved checkpoint (val_acc={val_acc:.4f})")

Phase 1: Training classification head (backbone frozen)


Epoch  1 | Train loss: 1.2496 acc: 0.5413 | Val loss: 1.0711 acc: 0.6212
           ✓ Saved checkpoint (val_acc=0.6212)


Epoch  2 | Train loss: 1.0535 acc: 0.6123 | Val loss: 1.0039 acc: 0.6338
           ✓ Saved checkpoint (val_acc=0.6338)


Epoch  3 | Train loss: 1.0002 acc: 0.6344 | Val loss: 0.9690 acc: 0.6475
           ✓ Saved checkpoint (val_acc=0.6475)


In [42]:
# --- Phase 2: Unfreeze backbone and fine-tune with lower LR ---
model.unfreeze_backbone()
optimizer = optim.AdamW(model.parameters(), lr=LR / 10, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - 3)

print("\nPhase 2: Full fine-tuning (backbone unfrozen)")
for epoch in range(3, EPOCHS):
    train_loss, train_acc, _, _ = run_epoch(model, train_loader, optimizer, train=True)
    val_loss, val_acc, _, _ = run_epoch(model, val_loader, train=False)
    scheduler.step()
    
    print(f"Epoch {epoch+1:2d} | Train loss: {train_loss:.4f} acc: {train_acc:.4f} | Val loss: {val_loss:.4f} acc: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT)
        print(f"           ✓ Saved checkpoint (val_acc={val_acc:.4f})")

Backbone unfrozen for full fine-tuning.

Phase 2: Full fine-tuning (backbone unfrozen)


Epoch  4 | Train loss: 0.6691 acc: 0.7546 | Val loss: 0.5407 acc: 0.8087
           ✓ Saved checkpoint (val_acc=0.8087)


Epoch  5 | Train loss: 0.4605 acc: 0.8328 | Val loss: 0.4575 acc: 0.8300
           ✓ Saved checkpoint (val_acc=0.8300)


Epoch  6 | Train loss: 0.3627 acc: 0.8700 | Val loss: 0.4192 acc: 0.8600
           ✓ Saved checkpoint (val_acc=0.8600)


Epoch  7 | Train loss: 0.2745 acc: 0.9010 | Val loss: 0.4149 acc: 0.8700
           ✓ Saved checkpoint (val_acc=0.8700)


Epoch  8 | Train loss: 0.1919 acc: 0.9325 | Val loss: 0.4038 acc: 0.8731
           ✓ Saved checkpoint (val_acc=0.8731)


Epoch  9 | Train loss: 0.1235 acc: 0.9594 | Val loss: 0.4025 acc: 0.8875
           ✓ Saved checkpoint (val_acc=0.8875)


Epoch 10 | Train loss: 0.0851 acc: 0.9733 | Val loss: 0.4053 acc: 0.8900
           ✓ Saved checkpoint (val_acc=0.8900)


## Step 6 — Final Test Evaluation

In [43]:
# Load best checkpoint
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))

_, test_acc, test_preds, test_labels = run_epoch(model, test_loader, train=False)

print(f"Test Accuracy: {test_acc:.4f}\n")
print(classification_report(test_labels, test_preds, target_names=NEW_CLASS_NAMES))

Test Accuracy: 0.8818

              precision    recall  f1-score   support

       angry       0.88      0.87      0.87       297
     disgust       0.78      0.74      0.76        98
        fear       0.84      0.87      0.86       144
       happy       0.97      0.97      0.97       407
     natural       0.75      0.78      0.76       136
         sad       0.87      0.88      0.88       298
   surprised       0.90      0.87      0.88       219

    accuracy                           0.88      1599
   macro avg       0.86      0.85      0.85      1599
weighted avg       0.88      0.88      0.88      1599



## Step 7 — Export logits for multimodal fusion
When combining image + audio + text predictions, you'll want raw logits (or softmax probabilities) from each modality.

In [44]:
import numpy as np

def extract_probabilities(model, loader):
    """Returns (softmax probabilities, true labels) for a DataLoader."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Extracting probs"):
            images = images.to(device)
            logits = model(images)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_labels.extend(labels.numpy())
    return np.vstack(all_probs), np.array(all_labels)

test_probs, test_true = extract_probabilities(model, test_loader)
print(f"Probabilities shape: {test_probs.shape}")  # (N, 7)

# Save for fusion
np.save("image_test_probs.npy", test_probs)
np.save("image_test_labels.npy", test_true)
print("Saved image_test_probs.npy and image_test_labels.npy for multimodal fusion.")

Extracting probs: 100%|██████████| 50/50 [01:01<00:00,  1.22s/it]

Probabilities shape: (1599, 7)
Saved image_test_probs.npy and image_test_labels.npy for multimodal fusion.


## DINOv2 Image Emotion Recognition — Results

### Training Strategy
The DINOv2 model was fine-tuned using a two-phase transfer learning approach:

- **Phase 1 (Frozen Backbone):** Only the lightweight classification head (LayerNorm + Linear, ~6,900 parameters) was trained while the DINOv2 backbone (86M parameters) remained frozen. This prevents the randomly-initialised head from corrupting the pretrained weights with large gradients early in training, allowing the head to first learn a stable mapping from DINOv2's visual features to the 7 emotion classes.
- **Phase 2 (Unfrozen Backbone):** Once the head stabilised, the full backbone was unfrozen and fine-tuned jointly at a reduced learning rate (LR / 10). This allows DINOv2 to subtly adapt its representations specifically for facial emotion recognition.

### Test Set Performance
The model achieved a **test accuracy of 88.18%** on 1,599 test samples across 7 emotion classes.

| Class | Precision | Recall | F1-Score | Support |
|:---|:---:|:---:|:---:|:---:|
| Angry | 0.88 | 0.87 | 0.87 | 297 |
| Disgust | 0.78 | 0.74 | 0.76 | 98 |
| Fear | 0.84 | 0.87 | 0.86 | 144 |
| Happy | 0.97 | 0.97 | 0.97 | 407 |
| Natural | 0.75 | 0.78 | 0.76 | 136 |
| Sad | 0.87 | 0.88 | 0.88 | 298 |
| Surprised | 0.90 | 0.87 | 0.88 | 219 |
| **Macro avg** | **0.86** | **0.85** | **0.85** | **1599** |
| **Weighted avg** | **0.88** | **0.88** | **0.88** | **1599** |

*Table: Per-class classification report on the test set.*

### Discussion
Overall, the model performs strongly across all 7 classes. **Happy** achieves near-perfect classification (F1 = 0.97), likely due to its visually distinct features. **Angry**, **Sad**, and **Surprised** also perform well (F1 = 0.87–0.88). The weakest classes are **Disgust** and **Natural** (F1 = 0.76), which is consistent with the broader literature as these expressions tend to be subtle and visually ambiguous.

The macro-averaged F1 of 0.85 is particularly noteworthy as it weights all classes equally regardless of support size, indicating robust performance even on smaller classes such as Disgust ($n=98$) and Fear ($n=144$).